# Phase 3 — Bronze Ingestion

**Objectifs:**
- [x] Lecture Parquet
- [x] Ingestion PySpark
- [x] Ajout Metadata
- [x] Delta Tables
- [x] Vérification des données

**Livrable:** Bronze Layer

**Rôle de la couche Bronze:** stocker les données **telles qu'elles arrivent** de la source (aucun nettoyage), en **Delta**, avec des **métadonnées de traçabilité**. Si une erreur est faite plus tard (Silver/Gold), on peut toujours tout recalculer depuis Bronze.

```
Volume raw/landing (Parquet, CSV)  ──►  workspace.bronze.yellow_trips / taxi_zones  (Delta + metadata)
```

## 0. Setup

In [1]:
from lakehouse_config import *
from lakehouse_transforms import *
spark = get_spark()
setup_schemas(spark)

from pyspark.sql import functions as F
from pyspark.sql.types import *
import pandas as pd

## 1. Lecture Parquet
### 1.1 Fichiers disponibles dans la zone d'atterrissage (landing)

In [2]:
w = get_workspace_client()
landing_files = sorted(f.name for f in w.files.list_directory_contents(RAW_VOLUME))
trip_files = [f for f in landing_files if f.startswith("yellow_tripdata_") and f.endswith(".parquet")]

print("Fichiers dans le Volume:", landing_files)
print("Fichiers trips a ingerer:", trip_files)

Fichiers dans le Volume: ['taxi_zone_lookup.csv', 'yellow_tripdata_2024-01.parquet']
Fichiers trips a ingerer: ['yellow_tripdata_2024-01.parquet']


### 1.2 Lecture
Parquet est un format **colonnaire** et **auto-décrit**: le schéma est stocké dans le fichier → pas besoin d'`inferSchema`.

In [3]:
trip_paths = [f"{RAW_VOLUME}/{f}" for f in trip_files]
raw_trips = spark.read.parquet(*trip_paths)

raw_zones = spark.read.csv(f"{RAW_VOLUME}/{ZONES_FILE}", header=True, inferSchema=True)

print(f"raw_trips: {raw_trips.count():,} lignes | raw_zones: {raw_zones.count():,} lignes")
raw_trips.printSchema()

raw_trips: 2,964,624 lignes | raw_zones: 265 lignes


root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)



## 2. Ingestion PySpark & 3. Ajout Metadata

On ajoute 3 colonnes techniques (préfixe `_` = colonne technique, pas métier):

| Colonne | Contenu | Utilité |
|---|---|---|
| `_source_file` | Chemin du fichier d'origine (`_metadata.file_path`) | Savoir **d'où vient** chaque ligne |
| `_ingestion_timestamp` | Date/heure de l'ingestion | Savoir **quand** elle est arrivée |
| `_batch_id` | Identifiant unique du chargement | Rejouer / annuler **un** chargement précis |

La fonction `add_ingestion_metadata()` est dans `lakehouse_transforms.py` (réutilisée en Phase 7).

In [4]:
import uuid

batch_id = str(uuid.uuid4())
bronze_trips = add_ingestion_metadata(raw_trips, batch_id)
bronze_zones = add_ingestion_metadata(raw_zones, batch_id)

print("batch_id:", batch_id)
bronze_trips.select("VendorID", "tpep_pickup_datetime", "total_amount", *METADATA_COLUMNS).show(3, truncate=False)

batch_id:

 78a6d09b-28bc-4911-8d7e-0d300e0bb9cf


+--------+--------------------+------------+-------------------------------------------------------------------+--------------------------+------------------------------------+
|VendorID|tpep_pickup_datetime|total_amount|_source_file                                                       |_ingestion_timestamp      |_batch_id                           |
+--------+--------------------+------------+-------------------------------------------------------------------+--------------------------+------------------------------------+
|2       |2024-01-01 00:57:55 |22.7        |dbfs:/Volumes/workspace/raw/landing/yellow_tripdata_2024-01.parquet|2026-11-08 19:02:46.069959|78a6d09b-28bc-4911-8d7e-0d300e0bb9cf|
|1       |2024-01-01 00:03:00 |18.75       |dbfs:/Volumes/workspace/raw/landing/yellow_tripdata_2024-01.parquet|2026-11-08 19:02:46.069959|78a6d09b-28bc-4911-8d7e-0d300e0bb9cf|
|1       |2024-01-01 00:17:06 |31.3        |dbfs:/Volumes/workspace/raw/landing/yellow_tripdata_2024-01.parquet|202

## 4. Delta Tables

**Delta Lake** = Parquet + un **journal de transactions** (`_delta_log`). Il apporte:
- **ACID**: une écriture réussit entièrement ou pas du tout.
- **Historique / Time travel**: chaque écriture crée une **version** consultable.
- **Schema enforcement**: refuse les données au mauvais schéma.
- **MERGE / UPDATE / DELETE** (utilisés en Phase 7).

### 4.1 Écriture (chargement initial complet)

In [5]:
(bronze_trips.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(T_BRONZE_TRIPS))

(bronze_zones.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(T_BRONZE_ZONES))

spark.sql(f"COMMENT ON TABLE {T_BRONZE_TRIPS} IS 'Bronze - NYC Yellow Taxi trips bruts (TLC) + metadata d ingestion'")
spark.sql(f"COMMENT ON TABLE {T_BRONZE_ZONES} IS 'Bronze - Referentiel des zones TLC brut + metadata d ingestion'")
print("Tables Bronze ecrites:", T_BRONZE_TRIPS, "|", T_BRONZE_ZONES)

Tables Bronze ecrites: workspace.bronze.yellow_trips | workspace.bronze.taxi_zones


### 4.2 Journal d'ingestion
On enregistre chaque fichier chargé dans `ops.ingestion_log`. La **Phase 7** s'en sert pour savoir quels fichiers sont **nouveaux**.

In [6]:
log_rows = (spark.table(T_BRONZE_TRIPS)
    .groupBy("_source_file", "_batch_id", "_ingestion_timestamp").count()
    .select(
        F.regexp_extract("_source_file", r"([^/]+)$", 1).alias("file_name"),
        F.lit(T_BRONZE_TRIPS).alias("target_table"),
        F.col("count").alias("row_count"),
        F.lit("SUCCESS").alias("status"),
        F.lit("full_load").alias("load_type"),
        F.col("_batch_id").alias("batch_id"),
        F.col("_ingestion_timestamp").alias("ingested_at"),
    ))

log_rows.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(T_INGESTION_LOG)
spark.table(T_INGESTION_LOG).show(truncate=False)

+-------------------------------+-----------------------------+---------+-------+---------+------------------------------------+--------------------------+
|file_name                      |target_table                 |row_count|status |load_type|batch_id                            |ingested_at               |
+-------------------------------+-----------------------------+---------+-------+---------+------------------------------------+--------------------------+
|yellow_tripdata_2024-01.parquet|workspace.bronze.yellow_trips|2964624  |SUCCESS|full_load|78a6d09b-28bc-4911-8d7e-0d300e0bb9cf|2026-11-08 19:02:50.517122|
+-------------------------------+-----------------------------+---------+-------+---------+------------------------------------+--------------------------+



### 4.3 Détails, historique et time travel

In [7]:
spark.sql(f"DESCRIBE DETAIL {T_BRONZE_TRIPS}") \
     .select("format", "name", "numFiles", "sizeInBytes", "createdAt", "lastModified").show(truncate=False)

+------+-----------------------------+--------+-----------+-----------------------+-------------------+
|format|name                         |numFiles|sizeInBytes|createdAt              |lastModified       |
+------+-----------------------------+--------+-----------+-----------------------+-------------------+
|delta |workspace.bronze.yellow_trips|1       |48170209   |2026-11-08 19:02:49.236|2026-11-08 19:03:01|
+------+-----------------------------+--------+-----------+-----------------------+-------------------+



In [8]:
spark.sql(f"DESCRIBE HISTORY {T_BRONZE_TRIPS}") \
     .select("version", "timestamp", "operation", "operationMetrics.numOutputRows").show(5, truncate=False)

+-------+-------------------+---------------------------------+-------------+
|version|timestamp          |operation                        |numOutputRows|
+-------+-------------------+---------------------------------+-------------+
|1      |2026-11-08 19:03:01|SET TBLPROPERTIES                |NULL         |
|0      |2026-11-08 19:02:56|CREATE OR REPLACE TABLE AS SELECT|2964624      |
+-------+-------------------+---------------------------------+-------------+



In [9]:
# Time travel: lire la table telle qu'elle etait a une version donnee
latest = spark.sql(f"DESCRIBE HISTORY {T_BRONZE_TRIPS}").agg(F.max("version")).collect()[0][0]
v = spark.read.option("versionAsOf", latest).table(T_BRONZE_TRIPS).count()
print(f"Version {latest}: {v:,} lignes")

Version 1: 2,964,624 lignes


## 5. Vérification des données

In [10]:
checks = []
bronze = spark.table(T_BRONZE_TRIPS)

# 1) Volumetrie: source == bronze
src_count, bronze_count = raw_trips.count(), bronze.count()
checks.append(("trips: nb lignes source == bronze", src_count, bronze_count, src_count == bronze_count))

src_z, bronze_z = raw_zones.count(), spark.table(T_BRONZE_ZONES).count()
checks.append(("zones: nb lignes source == bronze", src_z, bronze_z, src_z == bronze_z))

# 2) Schema: toutes les colonnes source presentes + 3 colonnes metadata
missing = set(raw_trips.columns) - set(bronze.columns)
checks.append(("colonnes source presentes", len(raw_trips.columns), len(raw_trips.columns) - len(missing), not missing))
checks.append(("colonnes metadata ajoutees", 3, len(set(METADATA_COLUMNS) & set(bronze.columns)),
               set(METADATA_COLUMNS) <= set(bronze.columns)))

# 3) Metadata jamais nulles
null_meta = bronze.filter(" OR ".join(f"{c} IS NULL" for c in METADATA_COLUMNS)).count()
checks.append(("metadata non nulles", 0, null_meta, null_meta == 0))

# 4) Total des montants identique (rien n'a ete modifie)
s1 = raw_trips.agg(F.round(F.sum("total_amount"), 2)).collect()[0][0]
s2 = bronze.agg(F.round(F.sum("total_amount"), 2)).collect()[0][0]
checks.append(("somme total_amount identique", s1, s2, s1 == s2))

pd.DataFrame(checks, columns=["controle", "attendu", "obtenu", "OK"])

,controle,attendu,obtenu,OK
0,trips: nb lignes source == bronze,2964624.00,2964624.00,True
1,zones: nb lignes source == bronze,265.00,265.00,True
2,colonnes source presentes,19.00,19.00,True
3,colonnes metadata ajoutees,3.00,3.00,True
4,metadata non nulles,0.00,0.00,True
5,somme total_amount identique,79456384.28,79456384.28,True


In [11]:
# Volumetrie par fichier source
bronze.groupBy("_source_file").count().show(truncate=False)

+-------------------------------------------------------------------+-------+
|_source_file                                                       |count  |
+-------------------------------------------------------------------+-------+
|dbfs:/Volumes/workspace/raw/landing/yellow_tripdata_2024-01.parquet|2964624|
+-------------------------------------------------------------------+-------+



## Conclusion — Bronze Layer

- **2 tables Delta** créées: `workspace.bronze.yellow_trips` (2 964 624 lignes) et `workspace.bronze.taxi_zones` (265 lignes).
- Les données sont **identiques à la source** (mêmes lignes, mêmes colonnes, même somme des montants) + **3 colonnes de traçabilité**.
- Chaque chargement est **historisé** (Delta history / time travel) et **journalisé** dans `ops.ingestion_log`.
- **Aucun nettoyage** ici: les anomalies identifiées en Phase 1 sont volontairement conservées → traitées en **Phase 4 (Silver)**.